# Messy sales data (synthetic)

**Applied AI Academy — problem set with worked answers**

**Used in:** Lesson 7 — AI runs on data plumbing — readiness beats model choice  
**Rows:** 260  ·  **Source file:** `messy_sales_data.csv`



---
### How to use this notebook
Run the cells in order. Each problem is stated first, then solved, then the
answer is spelled out. For teaching, hide the solution cells and let the room
attempt the task before revealing them.

> The data loads straight from `appliedai.center` — nothing to upload.


## Setup

In [ ]:
%matplotlib inline
import pandas as pd, numpy as np, os, urllib.request
import matplotlib, matplotlib.pyplot as plt
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

# Applied AI Academy chart style — calm, legible when projected
INK, ACCENT, CORE, BUILDER, DANGER = "#26214a", "#544d94", "#5b8a6e", "#a8845c", "#b05a5a"
matplotlib.rcParams.update({
    "figure.figsize": (9, 4.5), "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c9c4d8", "axes.labelcolor": INK, "axes.titlesize": 13,
    "axes.titleweight": "600", "axes.titlecolor": INK, "axes.titlepad": 14,
    "text.color": INK, "xtick.color": INK, "ytick.color": INK,
    "font.size": 10, "grid.color": "#e6e2ef", "axes.grid": True,
    "axes.axisbelow": True, "grid.linewidth": .8, "figure.facecolor": "white",
})

URL   = "https://appliedai.center/assets/datasets/messy_sales_data.csv"
LOCAL = "messy_sales_data.csv"
if not os.path.exists(LOCAL):
    print("Downloading", URL)
    urllib.request.urlretrieve(URL, LOCAL)

df = pd.read_csv(LOCAL, low_memory=False)
print("Shape:", df.shape)
df.head()

## What is in the file

In [ ]:
print("Columns and types:")
print(df.dtypes.to_string())
miss = df.isna().sum()
miss = miss[miss > 0]
print("\nMissing values:" if len(miss) else "\nNo missing values.")
if len(miss): print(miss.to_string())

---
## Problem 1 · Easy — describe

Find every data-quality problem before computing anything.

*Your turn. Write your answer in the cell below — the worked solution is in your instructor's edition.*

In [ ]:
# your code here


---
## Problem 2 · Intermediate — build

Clean it, and state every decision you made.

*Your turn. Write your answer in the cell below — the worked solution is in your instructor's edition.*

In [ ]:
# your code here


---
## Problem 3 · Advanced — judge

Now compute revenue — and show how wrong the naive number was.

*Your turn. Write your answer in the cell below — the worked solution is in your instructor's edition.*

In [ ]:
# your code here


---
## More visuals — the questions a room asks next

The summary above answers the three problems. This answers what gets asked
immediately afterwards: how each variable is actually distributed, what moves
with what, where the outcome concentrates, and how much of the total sits in how
few rows. The same six views appear in every dataset's notebook, so they can be
compared rather than relearned.

In [ ]:
def deep_visuals(df, target=None, cats=None, nums=None, title=""):
    '''Six more views of the same table.

    The single summary figure answers the three problems. This answers the
    questions a room asks next: how is each variable actually distributed, what
    moves with what, where does the target concentrate, and how much of the
    total sits in how few rows. Written once and configured per dataset, so
    every notebook shows the same six views and a learner can compare across
    datasets instead of relearning a new chart each time.
    '''
    import numpy as np, pandas as pd, matplotlib.pyplot as plt

    nums = nums or [c for c in df.select_dtypes("number").columns][:8]
    cats = cats or [c for c in df.select_dtypes("object").columns
                    if 1 < df[c].nunique() <= 12][:3]
    # A column named as numeric may still be stored as text (a rating written
    # "4 / 5", a salary with a thousands separator). Coerce, then keep only what
    # actually became numbers — otherwise the correlation step dies on a string.
    df = df.copy()
    keep = []
    for c in nums:
        if c not in df.columns:
            continue
        if not pd.api.types.is_numeric_dtype(df[c]):
            conv = pd.to_numeric(
                df[c].astype(str).str.replace(r"[^\d.\-]", "", regex=True).replace("", np.nan),
                errors="coerce")
            if conv.notna().mean() < .6:
                continue
            df[c] = conv
        keep.append(c)
    nums = keep
    cats = [c for c in cats if c in df.columns]

    fig = plt.figure(figsize=(13.5, 12))
    gs  = fig.add_gridspec(3, 3, hspace=.55, wspace=.32)
    if title:
        fig.suptitle(title, fontsize=15, color=INK, y=.985, fontweight="600")

    # 1 — distribution of the three most variable numeric columns
    show = sorted(nums, key=lambda c: df[c].std(ddof=0) / (abs(df[c].mean()) + 1e-9),
                  reverse=True)[:3]
    for i, c in enumerate(show):
        ax = fig.add_subplot(gs[0, i])
        v = df[c].dropna()
        # Real business data is long-tailed: one 3M outlier flattens every other
        # bar into the axis. Draw the 1st-99th percentile and say so, rather
        # than showing a chart whose only visible feature is the outlier.
        lo, hi = v.quantile([.01, .99])
        shown = v[(v >= lo) & (v <= hi)] if hi > lo else v
        ax.hist(shown, bins=min(30, max(8, int(np.sqrt(len(shown))))), color=ACCENT, alpha=.85)
        ax.axvline(v.median(), color=DANGER, lw=1.6, ls="--")
        ax.set_title(f"{c}", fontsize=11)
        ax.set_ylabel("rows" if i == 0 else "")
        ax.text(.97, .93, f"median {v.median():,.0f}", transform=ax.transAxes,
                ha="right", va="top", fontsize=8.5, color=DANGER)
        if len(shown) < len(v):
            ax.text(.97, .80, "1st-99th pct", transform=ax.transAxes,
                    ha="right", va="top", fontsize=7.5, color="#8b85a6")

    # 2 — what moves with what
    ax = fig.add_subplot(gs[1, :2])
    corr = df[nums].corr(numeric_only=True)
    im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(nums))); ax.set_xticklabels(nums, rotation=40, ha="right", fontsize=8.5)
    ax.set_yticks(range(len(nums))); ax.set_yticklabels(nums, fontsize=8.5)
    ax.grid(False); ax.set_title("Correlation — what moves with what")
    for i in range(len(nums)):
        for j in range(len(nums)):
            v = corr.iloc[i, j]
            if i != j and abs(v) >= .3:
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7.5,
                        color="white" if abs(v) > .6 else INK)
    fig.colorbar(im, ax=ax, fraction=.025, pad=.02)

    # 3 — the strongest single relationship, drawn
    ax = fig.add_subplot(gs[1, 2])
    pairs = [(abs(corr.iloc[i, j]), nums[i], nums[j])
             for i in range(len(nums)) for j in range(i + 1, len(nums))]
    if pairs:
        _, a, b = max(pairs)
        d = df[[a, b]].dropna()
        for col in (a, b):
            lo, hi = d[col].quantile([.01, .99])
            if hi > lo:
                d = d[(d[col] >= lo) & (d[col] <= hi)]
        if len(d) > 3000:
            d = d.sample(3000, random_state=0)
        ax.scatter(d[a], d[b], s=8, alpha=.35, color=ACCENT, edgecolors="none")
        if len(d) > 2:
            k, m = np.polyfit(d[a], d[b], 1)
            xs = np.linspace(d[a].min(), d[a].max(), 50)
            ax.plot(xs, k * xs + m, color=DANGER, lw=1.8)
        ax.set_xlabel(a); ax.set_ylabel(b)
        ax.set_title(f"Strongest pair (r={corr.loc[a, b]:.2f})", fontsize=11)

    # 4 — where the target concentrates
    ax = fig.add_subplot(gs[2, 0])
    if target and target in df.columns and cats:
        c = cats[0]
        if df[target].dropna().isin([0, 1, True, False]).all():
            r = df.groupby(c)[target].mean().mul(100).sort_values()
            ax.barh(r.index.astype(str), r.values, color=BUILDER)
            ax.set_xlabel("% positive"); ax.set_title(f"{target} rate by {c}", fontsize=11)
        else:
            grp = [df.loc[df[c] == k, target].dropna() for k in df[c].dropna().unique()[:8]]
            ax.boxplot(grp, vert=False, patch_artist=True,
                       boxprops=dict(facecolor="#e8e5f3", color=ACCENT),
                       medianprops=dict(color=DANGER))
            ax.set_yticklabels(list(df[c].dropna().unique()[:8]), fontsize=8.5)
            ax.set_title(f"{target} by {c}", fontsize=11)
    elif nums and cats:
        # No binary outcome to split by, so show how the headline number varies
        # across the main category — the comparison a room asks for anyway.
        c, n = cats[0], nums[0]
        g = df.groupby(c)[n].median().sort_values().tail(8)
        ax.barh(g.index.astype(str), g.values, color=BUILDER)
        ax.set_title(f"Median {n} by {c}", fontsize=11)
        for i, v in enumerate(g.values):
            ax.text(v, i, f" {v:,.0f}", va="center", fontsize=8)
    elif nums:
        v = df[nums[0]].dropna()
        lo, hi = v.quantile([.01, .99])
        ax.boxplot(v[(v >= lo) & (v <= hi)] if hi > lo else v, vert=False, patch_artist=True,
                   boxprops=dict(facecolor="#e8e5f3", color=ACCENT),
                   medianprops=dict(color=DANGER))
        ax.set_title(f"Spread of {nums[0]} (1st-99th pct)", fontsize=11)

    # 5 — how much sits in how few (Pareto)
    ax = fig.add_subplot(gs[2, 1])
    base = nums[0] if nums else None
    if base is not None and (df[base] >= 0).all():
        v = df[base].dropna().sort_values(ascending=False).values
        cum = np.cumsum(v) / v.sum() * 100
        pct = np.arange(1, len(v) + 1) / len(v) * 100
        ax.plot(pct, cum, color=ACCENT, lw=2)
        ax.axhline(80, color=DANGER, ls="--", lw=1.2)
        at80 = pct[np.argmax(cum >= 80)] if (cum >= 80).any() else 100
        ax.axvline(at80, color=DANGER, ls=":", lw=1.2)
        ax.set_xlabel("% of rows (largest first)"); ax.set_ylabel(f"% of {base}")
        ax.set_title(f"Concentration — 80% sits in {at80:.0f}% of rows", fontsize=11)

    # 6 — the categorical shape
    ax = fig.add_subplot(gs[2, 2])
    if cats:
        c = cats[-1]
        vc = df[c].value_counts().head(8).sort_values()
        ax.barh(vc.index.astype(str), vc.values, color=CORE)
        ax.set_title(f"Rows by {c}", fontsize=11)
        for i, v in enumerate(vc.values):
            ax.text(v, i, f" {v:,}", va="center", fontsize=8.5)
    plt.show()

deep_visuals(df, cats=["region","product"], nums=["unit_price","discount_pct"], title="Messy sales extract — six more views")